# movie-reviews

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/kaggle/working/eX-LAViTs")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/VuTrinhNguyenHoang/eX-LAViTs.git", str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print("Code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
%pip install -q timm==1.0.30 transformers==4.57.6 pyarrow==19.0.1 scikit-learn psutil

In [ ]:
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display
from src.models.linear_vit import create_linear_attention_vit
from src.utils.metrics import compute_loss_and_accuracy
from src.utils.revision_evaluation import (
    evaluate_methods,
    format_summary,
    save_metadata,
    benchmark_methods,
)

from transformers import BertTokenizerFast, AutoModel
from tqdm import tqdm
from src.models.linear_text_transformer import LinearTextTransformer
from src.utils.training import get_scheduler
import json


## Configuration

In [ ]:
DATASET = "movie-reviews"
MAX_LEN = 512
LR = 3e-05
WD = 0.01
EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 32
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print(NUM_WORKERS, DEVICE)
MAX_SAMPLES = 100
STEPS = 20
output_dir = Path("/kaggle/working/results") / DATASET
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / f"{DATASET}.pth"
print("GPU:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else None)


## Data

In [ ]:
class TextDataset(Dataset):

    def __init__(
        self, path, split: str, max_len: int, pad_token_id: int, use_rationale: bool = True
    ):
        df = pd.read_parquet(path)
        df = df[df["split"] == split].reset_index(drop=True)
        self.df = df
        self.max_len = max_len
        self.pad_token_id = pad_token_id
        self.use_rationale = use_rationale and "rationale_mask" in df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids = np.array(row["input_ids"]).astype(np.int64)
        label = int(row["label"])
        if self.use_rationale:
            rat = np.array(row["rationale_mask"]).astype(np.int64)
        else:
            rat = np.zeros_like(input_ids, dtype=np.int64)
        return {
            "input_ids": torch.from_numpy(input_ids),
            "label": torch.tensor(label, dtype=torch.long),
            "rationale_mask": torch.from_numpy(rat),
        }


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


In [ ]:
data_url = "/kaggle/input/ex-la-text/movie_rationales.parquet"
train_dataset = TextDataset(data_url, "train", MAX_LEN, pad_token_id=tokenizer.pad_token_id)
valid_dataset = TextDataset(
    data_url, "validation", MAX_LEN, pad_token_id=tokenizer.pad_token_id
)
test_dataset = TextDataset(data_url, "test", MAX_LEN, pad_token_id=tokenizer.pad_token_id)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
(len(train_loader), len(valid_loader), len(test_loader))


In [ ]:
batch_test = next(iter(train_loader))
num_classes = len(train_dataset.df["label"].unique())
print("Train / validation / test:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("Input shape:", batch_test["input_ids"].shape)
sample_ids = [f"test:{index}" for index in range(len(test_dataset))]
manifest = pd.DataFrame({"sample_id": sample_ids, "label": test_dataset.df["label"].tolist()})
manifest.to_csv(output_dir / "test_manifest.csv", index=False)


## Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(loader, leave=False, mininterval=20):
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.3)
        optimizer.step()
        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    return (avg_loss, acc)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids)
        loss = criterion(logits, labels)
        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    return (avg_loss, acc)


In [ ]:
from transformers import AutoModel


def init_and_freeze_token_embedding_from_bert(
    text_model, bert_name: str = "bert-base-uncased"
):
    bert = AutoModel.from_pretrained(bert_name)
    bert_embed = bert.embeddings.word_embeddings
    bert_vocab, bert_dim = (bert_embed.num_embeddings, bert_embed.embedding_dim)
    text_embed = text_model.token_embed
    txt_vocab, txt_dim = (text_embed.num_embeddings, text_embed.embedding_dim)
    with torch.no_grad():
        text_embed.weight.data.copy_(bert_embed.weight.data)
    text_embed.weight.requires_grad = False
    with torch.no_grad():
        bert_pos_w = bert.embeddings.position_embeddings.weight
        max_len_model = text_model.pos_embed.shape[1]
        text_model.pos_embed[:, 1:max_len_model, :].copy_(bert_pos_w[: max_len_model - 1])


In [ ]:
model = LinearTextTransformer(
    vocab_size=tokenizer.vocab_size,
    num_classes=2,
    max_seq_len=MAX_LEN,
    embed_dim=768,
    depth=4,
    num_heads=8,
    mlp_ratio=4.0,
    drop_rate=0.2,
    attn_drop_rate=0.0,
    drop_path_rate=0.2,
    mlp_drop_rate=0.1,
    kernel="elu",
    eps=1e-06,
).to(DEVICE)
init_and_freeze_token_embedding_from_bert(model)


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=WD
)
scheduler = get_scheduler(optimizer, "cosine", **{"T_max": EPOCHS, "eta_min": 0})


In [ ]:
history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
no_improve = 0
best_epoch = None
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc = evaluate(model, valid_loader, criterion, DEVICE)
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(
        f"[Epoch {epoch:02d}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )
    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        best_epoch = epoch
        torch.save(
            {
                "model_state_dict": best_state,
                "epoch": best_epoch,
                "best_val_loss": best_val_loss,
            },
            checkpoint_path,
        )
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break
pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)


In [ ]:
model.load_state_dict(best_state, strict=True)
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
classification = dict(loss=test_loss, accuracy=test_acc, best_epoch=best_epoch)
print(classification)
(output_dir / "classification.json").write_text(json.dumps(classification, indent=2))
del best_state, optimizer, scheduler, batch_test
model.eval()
model.zero_grad(set_to_none=True)
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
import matplotlib.pyplot as plt

epochs = history["epoch"]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="train_acc")
plt.plot(epochs, history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Evaluation

In [ ]:
config = dict(
    dataset=DATASET,
    seed=SEED,
    max_length=MAX_LEN,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WD,
    label_smoothing=0.1,
    grad_clip_norm=1.3,
    learning_rate_schedule="constant; original cosine scheduler constructed but not stepped",
    initialization="BERT token and position embeddings; token embeddings frozen; transformer blocks trained from scratch",
    validation_test_overlap=False,
    split_protocol="original notebook unchanged",
    data_path=data_url,
    split_sizes=dict(
        train=len(train_dataset), validation=len(valid_dataset), test=len(test_dataset)
    ),
    model_config=dict(
        embed_dim=768,
        depth=4,
        num_heads=8,
        mlp_ratio=4.0,
        drop_rate=0.2,
        attn_drop_rate=0.0,
        drop_path_rate=0.2,
        mlp_drop_rate=0.1,
    ),
)
metadata = save_metadata(output_dir, config, model, checkpoint_path)


In [ ]:
summary = evaluate_methods(
    model,
    test_dataset,
    DEVICE,
    output_dir,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    pad_id=tokenizer.pad_token_id,
    sample_ids=sample_ids,
)
display(format_summary(summary))


## Compute

In [ ]:
xb = test_dataset[0]["input_ids"].unsqueeze(0).to(DEVICE)
compute = benchmark_methods(model, xb, output_dir, text=True)
display(compute)
print("Saved:", output_dir)
